In [ ]:
import platform
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import seaborn as sns

system_name = platform.system()
if system_name == 'Darwin':
    plt.rcParams['font.family'] = 'AppleGothic'
elif system_name == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
else:
    nanum_fonts = [f.name for f in fm.fontManager.ttflist if 'Nanum' in f.name]
    plt.rcParams['font.family'] = nanum_fonts[0] if nanum_fonts else 'DejaVu Sans'

plt.rcParams['axes.unicode_minus'] = False
sns.set_theme(style='whitegrid', font=plt.rcParams['font.family'])

print(f'환경 설정 완료! 적용된 폰트: {plt.rcParams["font.family"]}')

In [ ]:
df = pd.read_csv("../data/18_열처리.csv")
print(df.shape)
df.info()
print(df.isna().sum().sum())

### 실습 1. 첫 분포 그래프 그리기
- 목표: histplot으로 소입로온도 분포를 그리고 중심·퍼짐 확인
- 단계: ① histplot으로 분포 그리기 -> ② 평균·표준편차로 중심·퍼짐 수치 확인
- 예상 결과: 평균 859.42 근처에 몰린 종 모양 분포, 표준편차 약 2.0

In [ ]:
print(round(df["소입로온도"].mean(), 2), round(df["소입로온도"].std(), 2))  # 859.42 2.0
sns.histplot(data=df, x="소입로온도")
plt.title("소입로온도 분포")
plt.xlabel("소입로온도(℃)")
plt.show()

### 실습 2. bins·kde 조절
- 목표: bins 값을 바꿔가며 분포 모양 변화를 관찰하고 kde 곡선 추가
- 단계: ① bins=17(기본 감) -> ② bins=5·40 비교 -> ③ kde=True로 곡선 얹기
- 예상 결과: bins=17에서 봉우리와 퍼짐이 가장 또렷하게 보임

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
sns.histplot(data=df, x="소입로온도", bins=5, ax=axes[0])
axes[0].set_title("bins=5")
sns.histplot(data=df, x="소입로온도", bins=17, kde=True, ax=axes[1])
axes[1].set_title("bins=17 + kde")
sns.histplot(data=df, x="소입로온도", bins=40, ax=axes[2])
axes[2].set_title("bins=40")
plt.show()

### 실습 3. 정상·이상 분포 비교
- 목표: hue로 판정별 소입로온도 분포를 색으로 겹쳐 비교
- 단계: ① x=소입로온도, hue=판정으로 histplot 그리기 -> ② 정상·이상 평균 비교
- 예상 결과: 이상 그룹 평균(856.88)이 정상 그룹(859.53)보다 뚜렷이 낮음

In [ ]:
mean_by_verdict = df.groupby("판정")["소입로온도"].mean().round(2)
print(mean_by_verdict)  # 이상 856.88, 정상 859.53
sns.histplot(data=df, x="소입로온도", hue="판정", kde=True)
plt.title("판정별 소입로온도 분포")
plt.show()

### 실습 4. 여러 센서 분포 한눈에 보기
- 목표: 반복문으로 소입로온도·제어출력·건조출력 분포를 차례로 그려 비교
- 단계: ① 컬럼 목록을 반복하며 각 분포를 subplot에 그리기 -> ② 제목 달기
- 예상 결과: 세 센서의 분포 모양(중심·퍼짐)을 나란히 확인

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
sensor_cols = ["소입로온도", "제어출력", "건조출력"]
for ax, col in zip(axes, sensor_cols):
    sns.histplot(data=df, x=col, kde=True, ax=ax)
    ax.set_title(f"{col} 분포")
plt.tight_layout()
plt.show()

### 실습 5. 단일 센서 박스플롯
- 목표: boxplot으로 소입로온도의 사분위수와 이상치를 확인
- 단계: ① Q1·Q3·IQR·경계 직접 계산 -> ② boxplot으로 같은 값이 자동 표시되는지 확인
- 예상 결과: IQR 1.4, 경계 856.7~862.3, 이 범위를 벗어난 이상치 32건(10.77%)

In [ ]:
q1 = df["소입로온도"].quantile(0.25)
q3 = df["소입로온도"].quantile(0.75)
iqr = q3 - q1
lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
print(round(iqr, 2), round(lower, 2), round(upper, 2))  # 1.4 856.7 862.3
mask = (df["소입로온도"] < lower) | (df["소입로온도"] > upper)
print(mask.sum(), round(mask.mean() * 100, 2))  # 32 10.77
sns.boxplot(data=df, y="소입로온도")
plt.title("소입로온도 박스플롯")
plt.ylabel("소입로온도(℃)")
plt.show()

### 실습 6. 라인별·판정별 박스플롯 비교
- 목표: x=라인, hue=판정으로 교대조별·판정별 소입로온도 분포를 한 번에 비교
- 단계: ① x=라인만으로 라인별 상자 비교 -> ② hue=판정을 더해 이중 비교
- 예상 결과: 세 라인 평균이 859.20~859.64로 큰 차이는 없음 - 라인 자체보다 판정 차이에 주목

In [ ]:
line_mean = df.groupby("라인")["소입로온도"].mean().round(2)
print(line_mean)  # 야간859.64, 주간859.20, 특근859.43
sns.boxplot(data=df, x="라인", y="소입로온도", hue="판정")
plt.title("라인별·판정별 소입로온도 박스플롯")
plt.ylabel("소입로온도(℃)")
plt.show()

### 실습 7. countplot·barplot 비교
- 목표: 개수는 countplot, 평균은 barplot으로 라인을 비교
- 단계: ① countplot으로 라인별 개수 세기 -> ② barplot으로 라인별 평균 제어출력 비교
- 예상 결과: 개수는 100/99/98로 균등, 평균 제어출력은 라인마다 다름

In [ ]:
line_control = df.groupby("라인")["제어출력"].mean().round(2)
print(line_control)
sns.countplot(data=df, x="라인", hue="라인", palette="pastel", legend=False)
plt.title("라인별 개수")
plt.show()

sns.barplot(data=df, x="라인", y="제어출력", hue="라인", palette="Spectral", legend=False)
plt.title("라인별 평균 제어출력")
plt.show()

### 실습 8. 라인별 정상·이상 비율 비교
- 목표: countplot(hue=판정)과 crosstab으로 라인별 정상·이상 개수를 그래프와 표로 정리
- 단계: ① countplot에 hue=판정 추가 -> ② groupby+unstack(또는 crosstab)으로 표 만들기
- 예상 결과: 주간이 이상 7건으로 세 라인 중 가장 많음(야간3, 특근2)

In [ ]:
cross = pd.crosstab(df["라인"], df["판정"])
print(cross)
sns.countplot(data=df, x="라인", hue="판정")
plt.title("라인별 판정 개수")
plt.show()